# Regresión lineal simple y múltiple paso a paso

Este notebook está pensado para una clase introductoria de econometría aplicada a estudiantes de negocios.

## Contexto del ejemplo
Tenemos información de ejecutivos comerciales. Queremos explicar las **ventas mensuales** a partir de tres variables:

- **Gasto_Marketing**: inversión mensual asignada a campañas.
- **Horas_Capacitacion**: horas de capacitación comercial del mes.
- **Experiencia_Anos**: años de experiencia del ejecutivo.
- **Ventas_Mensuales**: variable dependiente.

Trabajaremos en tres etapas:

1. **Regresión lineal simple** manualmente.
2. **Regresión lineal múltiple** usando la fórmula matricial.
3. **Regresión múltiple** usando librerías de Python.

In [ ]:
import pandas as pd
import numpy as np

## 1. Cargar la base de datos

La base fue guardada en formato Excel para que también la puedas mostrar en clases.

In [ ]:
df = pd.read_excel("base_regresion_negocios.xlsx")
df.head()

## 2. Mirar la estructura de los datos

In [ ]:
print(df.shape)
df.info()

In [ ]:
df.describe(numeric_only=True)

# Parte A. Regresión lineal simple

Primero estimaremos el modelo:

\[
Y_i = \beta_0 + \beta_1 X_i + u_i
\]

donde:

- \(Y_i\) = `Ventas_Mensuales`
- \(X_i\) = `Gasto_Marketing`

La idea es mostrar el cálculo **a mano**, usando:

\[
\hat\beta_1 = \frac{\operatorname{cov}(Y,X)}{\operatorname{var}(X)}
\]

\[
\hat\beta_0 = \bar Y - \hat\beta_1 \bar X
\]

In [ ]:
y = df["Ventas_Mensuales"]
x = df["Gasto_Marketing"]

y_mean = y.mean()
x_mean = x.mean()

cov_yx = np.cov(y, x, ddof=1)[0,1]
var_x = np.var(x, ddof=1)

b1 = cov_yx / var_x
b0 = y_mean - x_mean * b1

print("Media de Y:", round(y_mean, 4))
print("Media de X:", round(x_mean, 4))
print("Cov(Y,X):", round(cov_yx, 4))
print("Var(X):", round(var_x, 4))
print("b1 estimado:", round(b1, 4))
print("b0 estimado:", round(b0, 4))

## 3. Ecuación estimada

Una vez estimados \(\hat\beta_0\) y \(\hat\beta_1\), la recta ajustada queda:

In [ ]:
print(f"Ventas_Mensuales = {b0:.4f} + {b1:.4f} * Gasto_Marketing")

## 4. Valores ajustados y residuos

Recordemos que:

\[
\hat Y_i = \hat\beta_0 + \hat\beta_1 X_i
\]

y el residuo es:

\[
\hat u_i = Y_i - \hat Y_i
\]

In [ ]:
df["Y_hat_simple"] = b0 + b1 * df["Gasto_Marketing"]
df["Residuo_simple"] = df["Ventas_Mensuales"] - df["Y_hat_simple"]

df[["ID", "Ventas_Mensuales", "Gasto_Marketing", "Y_hat_simple", "Residuo_simple"]].head(10)

## 5. Visualización de la regresión simple

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))
plt.scatter(df["Gasto_Marketing"], df["Ventas_Mensuales"])
plt.plot(df["Gasto_Marketing"], df["Y_hat_simple"])
plt.xlabel("Gasto en marketing")
plt.ylabel("Ventas mensuales")
plt.title("Regresión lineal simple")
plt.grid(alpha=0.3)
plt.show()

# Parte B. Regresión lineal múltiple por matrices

Ahora queremos estimar:

\[
Y_i = \beta_0 + \beta_1 X_{1i} + \beta_2 X_{2i} + \beta_3 X_{3i} + u_i
\]

donde:

- \(Y\) = `Ventas_Mensuales`
- \(X_1\) = `Gasto_Marketing`
- \(X_2\) = `Horas_Capacitacion`
- \(X_3\) = `Experiencia_Anos`

La fórmula matricial de MCO es:

\[
\hat\beta = (X'X)^{-1}X'Y
\]

Aquí la matriz \(X\) **sí debe incluir una columna de unos** para el intercepto.

In [ ]:
Y = df["Ventas_Mensuales"].to_numpy().reshape(-1,1)

X = np.column_stack([
    np.ones(len(df)),
    df["Gasto_Marketing"].to_numpy(),
    df["Horas_Capacitacion"].to_numpy(),
    df["Experiencia_Anos"].to_numpy()
])

print("Dimensión de Y:", Y.shape)
print("Dimensión de X:", X.shape)
print()
print("Primeras 5 filas de X:")
print(X[:5,:])
print()
print("Primeras 5 filas de Y:")
print(Y[:5,:])

## 6. Construcción paso a paso de \(X'X\), \((X'X)^{-1}\) y \(X'Y\)

In [ ]:
XtX = X.T @ X
XtX_inv = np.linalg.inv(XtX)
XtY = X.T @ Y
beta_hat = XtX_inv @ XtY

print("Matriz X'X:")
print(np.round(XtX, 4))
print()
print("Matriz (X'X)^(-1):")
print(np.round(XtX_inv, 6))
print()
print("Matriz X'Y:")
print(np.round(XtY, 4))
print()
print("Vector beta estimado:")
print(np.round(beta_hat, 4))

## 7. Interpretación de los coeficientes

El vector \(\hat\beta\) está ordenado así:

\[
\hat\beta =
\begin{bmatrix}
\hat\beta_0 \\
\hat\beta_1 \\
\hat\beta_2 \\
\hat\beta_3
\end{bmatrix}
\]

es decir:

- \(\hat\beta_0\): intercepto
- \(\hat\beta_1\): efecto marginal de `Gasto_Marketing`
- \(\hat\beta_2\): efecto marginal de `Horas_Capacitacion`
- \(\hat\beta_3\): efecto marginal de `Experiencia_Anos`

In [ ]:
b0_m, b1_m, b2_m, b3_m = beta_hat.flatten()

print(f"Intercepto (b0): {b0_m:.4f}")
print(f"Coeficiente de Gasto_Marketing (b1): {b1_m:.4f}")
print(f"Coeficiente de Horas_Capacitacion (b2): {b2_m:.4f}")
print(f"Coeficiente de Experiencia_Anos (b3): {b3_m:.4f}")

## 8. Predicción manual con la forma matricial

Una vez estimado \(\hat\beta\), los valores ajustados se obtienen con:

\[
\hat Y = X\hat\beta
\]

y los residuos con:

\[
\hat u = Y - \hat Y
\]

In [ ]:
Y_hat_matriz = X @ beta_hat
residuos_matriz = Y - Y_hat_matriz

df["Y_hat_multiple_matriz"] = Y_hat_matriz.flatten()
df["Residuo_multiple_matriz"] = residuos_matriz.flatten()

df[[
    "ID",
    "Ventas_Mensuales",
    "Y_hat_multiple_matriz",
    "Residuo_multiple_matriz"
]].head(10)

# Parte C. Regresión múltiple con librerías de Python

Ahora repetimos la misma estimación, pero usando `statsmodels`, que es muy útil para econometría porque entrega:

- coeficientes,
- errores estándar,
- estadísticos t,
- p-values,
- \(R^2\),
- intervalos de confianza.

In [ ]:
import statsmodels.api as sm

In [ ]:
X_lib = df[["Gasto_Marketing", "Horas_Capacitacion", "Experiencia_Anos"]]
X_lib = sm.add_constant(X_lib)

modelo = sm.OLS(df["Ventas_Mensuales"], X_lib).fit()
print(modelo.summary())

## 9. Comparación entre el método matricial y el método con librerías

Los coeficientes deberían coincidir, salvo pequeñas diferencias por redondeo.

In [ ]:
comparacion = pd.DataFrame({
    "Coef_Matricial": beta_hat.flatten(),
    "Coef_Statsmodels": modelo.params.values
}, index=["const", "Gasto_Marketing", "Horas_Capacitacion", "Experiencia_Anos"])

comparacion["Diferencia"] = comparacion["Coef_Matricial"] - comparacion["Coef_Statsmodels"]
comparacion

# Parte D. Ideas para explicar en clase

## Regresión simple
La regresión simple sirve para mostrar la lógica básica:
- una variable dependiente,
- una variable explicativa,
- pendiente e intercepto,
- ajuste y residuos.

## Regresión múltiple
La regresión múltiple permite responder una pregunta más realista en negocios:
> ¿Cómo cambian las ventas cuando cambia una variable, manteniendo constantes las otras?

Eso hace que la interpretación sea más poderosa.

## Mensaje pedagógico
- La fórmula manual ayuda a entender **de dónde salen los coeficientes**.
- La forma matricial ayuda a entender **cómo se generaliza el problema**.
- Las librerías ayudan a trabajar en aplicaciones reales con más rapidez y más inferencia estadística.

# Ejercicio sugerido para los estudiantes

1. Repetir la regresión simple usando `Horas_Capacitacion` como única X.
2. Repetir la regresión simple usando `Experiencia_Anos` como única X.
3. Comparar cuál regresión simple parece explicar mejor las ventas.
4. Revisar si en la regresión múltiple los signos de los coeficientes coinciden con la intuición de negocios.
5. Discutir por qué los coeficientes de la regresión múltiple no tienen que ser iguales a los de las regresiones simples.